# 03 · Configuração 3: especialização com LoRA
O LoRA ensina **comportamento** (JSON, citar o id certo, recusar quando o trecho não está no
contexto), não decora a lei. Treinado só com `avaliacao/lora_treino.jsonl` (disjunto do teste).
Para cada pergunta que consta geramos também um exemplo **sem** o trecho certo, cujo alvo é a recusa.

In [1]:
# Preparação do ambiente (Colab ou local)
import os, sys, subprocess
REPO_URL = "https://github.com/giuliano10/GS-GENENERATIVE-AI.git"
SUBPASTA = "assistente-acessibilidade-calcadas"   # pasta do projeto dentro do repositório

if "google.colab" in sys.modules:
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", REPO_URL, "/content/repo"], check=True)
    else:   # mesma sessão de outro notebook: atualiza o código
        subprocess.run(["git", "-C", "/content/repo", "pull", "-q"])
    os.chdir(os.path.join("/content/repo", SUBPASTA))
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError("Falha no pip install (detalhes acima)")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Pasta do projeto:", os.getcwd())

Pasta do projeto: /content/repo/assistente-acessibilidade-calcadas


In [2]:
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "nenhuma (use Ambiente de execução > Alterar tipo > T4 GPU)")

GPU: Tesla T4


In [3]:
from src.base_conhecimento import carregar_trechos
from src.avaliacao import carregar_jsonl
from src.busca import Buscador, carregar_limiar
from src.config import ARQ_TESTE, ARQ_TREINO
from src.lora import montar_exemplos, treinar
trechos = carregar_trechos(); por_id = {t["id"]: t for t in trechos}
treino, teste = carregar_jsonl(ARQ_TREINO), carregar_jsonl(ARQ_TESTE)
buscador = Buscador(trechos, modo="hibrido")
exemplos = montar_exemplos(treino, buscador, por_id)
print(len(exemplos), "exemplos de treino")
print(exemplos[0]["mensagens"][1]["content"][:600], "\n\nALVO:", exemplos[0]["alvo"])

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


109 exemplos de treino
TRECHOS:
[id: SP-L16673-art1] (Lei Municipal nº 16.673/2017 (Estatuto do Pedestre – São Paulo), Art. 1)
Art. 1º Fica instituído no Município de São Paulo o Estatuto do Pedestre.
---
[id: SP-L16673-art8-p2] (Lei Municipal nº 16.673/2017 (Estatuto do Pedestre – São Paulo), Art. 8 (parte 2))
Art. 8º O Estatuto do Pedestre tem os seguintes objetivos: [...]
XII - a melhoria das condições de saúde da população pela prática da atividade física da caminhada;
XIII - a conscientização da sociedade quanto aos efeitos indesejáveis da utilização do veículo automotor nas locomoções urbanas;
XIV - o incentiv 

ALVO: {"resposta": "Não consta na base de conhecimento.", "fontes": [], "recusa": true}


In [4]:
treinar(exemplos, epocas=3, lr=2e-4, r=16)

`torch_dtype` is deprecated! Use `dtype` instead!


trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


The model is already on multiple devices. Skipping the move to device specified in `args`.
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


Step,Training Loss
5,0.345100
10,0.292400
15,0.199200
20,0.174700
25,0.182500
30,0.215200
35,0.140500
40,0.165400


Adaptador salvo em /content/repo/assistente-acessibilidade-calcadas/modelos/lora-acessibilidade


## Avaliação com o adaptador

In [5]:
import gc, torch
gc.collect(); torch.cuda.empty_cache()
from src.gerador import carregar_modelo
from src.experimento import rodar
modelo = carregar_modelo(com_lora=True); print(modelo.nome)
rag_lora = rodar("rag_lora", teste, modelo, buscador, carregar_limiar())
for s in rag_lora[:3]: print(s["pergunta"], "\n ->", s["bruto"][:300], "\n")

Qwen/Qwen2.5-1.5B-Instruct + LoRA(/content/repo/assistente-acessibilidade-calcadas/modelos/lora-acessibilidade)
[rag_lora] 10/60
[rag_lora] 20/60
[rag_lora] 30/60
[rag_lora] 40/60
[rag_lora] 50/60
[rag_lora] 60/60
Qual a largura mínima da faixa livre da calçada em São Paulo? 
 -> {"resposta": "Não consta na base de conhecimento.", "fontes": [], "recusa": true} 

Qual a inclinação transversal máxima permitida na faixa livre da calçada? 
 -> {"resposta": "Não consta na base de conhecimento.", "fontes": [], "recusa": true} 

Qual a largura mínima da faixa de serviço da calçada? 
 -> {"resposta": "Não consta na base de conhecimento.", "fontes": [], "recusa": true} 



O adaptador fica em `modelos/lora-acessibilidade/` (~20 MB) e é versionado no repositório.